In [5]:
from pathlib import Path

import io
import onnx
import torch
from onnxsim import simplify

import sys
sys.path.append('..')
from model import Model

In [6]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

### Generating forward-only graph

Number of classes:
- cifar10: 10
- cifar100: 100
- flowers: 102
- food: 101
- pets: 37

In [ ]:
# Create a classifier instance
device = "cpu"
batch_size = 8
num_classes = 102
channels = 3
img_h, img_w = 128, 128

model_name = 'resnet10t' # resnet18, resnet34, mobilenetv2_100, efficientnet_b2, visformer_tiny

model_frozen_layer_num = {
    'resnet18': 12,
    'resnet34': 20,
    'mobilenetv2_100': 21,
    'efficientnet_b2': 27,
    'visformer_tiny': 24,
    'lcnet_100': 19,
    'semnasnet_100': 20,
    'mobilenetv3_large_100': 22,
    'resnet10t': 14,
}
frozen_layer_num = model_frozen_layer_num[model_name]

# artifacts path
artifacts_path = Path(f'../artifacts/{model_name}/frozen_layer_{frozen_layer_num}_classes_{num_classes}')
artifacts_path.mkdir(parents=True, exist_ok=True)

pt_model = Model(model_name=model_name, num_classes=num_classes, channels=channels, frozen_layer_num=frozen_layer_num, img_size=img_h)
print(f'Model trainable parameters: {count_parameters(pt_model)}')

for param in pt_model.named_parameters():
    if 'frozen_features' in param[0] or 'bn' in param[0] or 'downsample.1' in param[0]:
        param[1].requires_grad = False
    else:
        param[1].requires_grad = True

torch.save(pt_model, artifacts_path / f'{model_name}.pth')
pt_model = torch.load(artifacts_path / f'{model_name}.pth', weights_only=False)
for param in pt_model.named_parameters():
    if param[1].requires_grad:
        print(param[0].ljust(40), '->\t', param[1].requires_grad)

print(f'Model trainable parameters: {count_parameters(pt_model)}')

Number of layers: 16
Requires grad:
	1.   Conv2d:                   True	->	frozen
	2.   BatchNorm2d:              True	->	frozen
	3.   ReLU:                     False	->	frozen
	4.   Conv2d:                   True	->	frozen
	5.   BatchNorm2d:              True	->	frozen
	6.   ReLU:                     False	->	frozen
	7.   Conv2d:                   True	->	frozen
	8.   BatchNorm2d:              True	->	frozen
	9.   ReLU:                     False	->	frozen
	10.  MaxPool2d:                False	->	frozen
	11.  BasicBlock:               True	->	frozen
	12.  BasicBlock:               True	->	frozen
	13.  BasicBlock:               True	->	frozen
	14.  BasicBlock:               True	->	frozen
	15.  SelectAdaptivePool2d:     False	->	frozen
	16.  Linear:                   True	->	trainable
Number of layers requiring grad: 11
Number of trainable blocks: 1
Model trainable parameters: 4974814
output.weight                            ->	 True
output.bias                              ->	 True
Mo

### Generating forward-only graph

In [8]:
# Generate a random input.
model_inputs = (torch.randn(batch_size, channels, img_h, img_w, device=device),)

model_outputs = pt_model(*model_inputs)
if isinstance(model_outputs, torch.Tensor):
    model_outputs = [model_outputs]
    
input_names = ["input"]
output_names = ["output"]
dynamic_axes = {"input": {0: "batch_size"}, "output": {0: "batch_size"}}

f = io.BytesIO()
torch.onnx.export(
    pt_model,
    model_inputs,
    f,
    input_names=input_names,
    output_names=output_names,
    opset_version=17,
    do_constant_folding=False,
    # training=torch.onnx.TrainingMode.PRESERVE,
    dynamic_axes=dynamic_axes,
    export_params=True,
    # keep_initializers_as_inputs=False,
)

onnx_model = onnx.load_model_from_string(f.getvalue())
print(f'Simplifying ONNX {model_name} model...')
eval_model, check = simplify(onnx_model)
onnx.save(onnx_model, artifacts_path / f'{model_name}.onnx')

/tmp/ipykernel_826148/3754891360.py:13: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(


Simplifying ONNX resnet10t model...
